In [1]:
!mkdir -p ~/.kaggle && echo KGAT_6b26e673b17306239bfd24c6431c2325 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [2]:
!kaggle competitions download -c state-farm-distracted-driver-detection

100% 4.00G/4.00G [00:57<00:00, 75.4MB/s]



In [ ]:
!unzip /content/state-farm-distracted-driver-detection.zip

Importing Necessary Packages

In [5]:
import glob
import os
import random
import shutil
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.applications import vgg16,mobilenet_v3
from keras import layers
from keras.utils import plot_model
from keras import models,optimizers
from keras.callbacks import EarlyStopping
from keras.applications.imagenet_utils import preprocess_input
from keras.callbacks import EarlyStopping
from keras.applications import resnet50
import keras
import numpy as np

In [6]:
# path = "/kaggle/input/state-farm-distracted-driver-detection/imgs/"
path = "/content/imgs/"
train_dir = path + "train/"
valid_dir = path + "val/"
test_dir  = path + 'test/' # so here we are tr

In [7]:
classes = [c for c in os.listdir(train_dir)]
classes.sort()
print(classes)

['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


In [ ]:
classes.sort()
for c in classes:

    fig=plt.figure(figsize=(20,20))
    cl = train_dir + c
    for i,sample in enumerate(random.sample(os.listdir(train_dir + c) , 10)):
        fig.add_subplot(10, 10, i+1)
        image = plt.imread(cl + '/' + sample)

        plt.imshow(image)
        plt.xticks([])
        plt.yticks([])
        plt.title(f"{c}")

Mapping the classes for to make excel sheet at the end of the project.

In [9]:
classes_mapping = {   'c0' : "safe_driving",
                      'c1' : "texting-right",
                      'c2' : "talking_on_the_phone-right",
                      'c3' : "texting-left",
                      'c4' : "talking_on_the_phone-left",
                      'c5' : "operating_the_radio",
                      'c6' : "drinking",
                      'c7' : "reaching_behind",
                      'c8' : "hair-and-makeup",
                      'c9' : "talking_to_passenger"}

In [10]:
for c in os.listdir(train_dir):
    shutil.move(os.path.join(train_dir,c), os.path.join(train_dir,classes_mapping[f'{c}']))

here we can see that different lenghts for validation , test and train data.

In [ ]:
for c in os.listdir(train_dir):
    os.makedirs(valid_dir + '/' + c, exist_ok=True)
    os.makedirs(test_dir + '/' + c, exist_ok=True)

    c_train_dir = train_dir + c
    c_len = len([sample for sample in os.listdir(c_train_dir)])
    print(c_len)

    for sample in random.sample(os.listdir(c_train_dir) , int(float(0.1) * c_len)):
        shutil.move(c_train_dir + '/' + sample, valid_dir + c)

    for sample in random.sample(os.listdir(c_train_dir) , int(float(0.1) * c_len)):
        shutil.move(c_train_dir + '/' + sample, test_dir + c)

Data Augmentation

In [12]:
train_datagen_augmentation = ImageDataGenerator(rescale=1 / 255.0,
                                                zoom_range=0.05,
                                                width_shift_range=0.05,
                                                height_shift_range=0.05,
                                                shear_range=0.05,
                                                fill_mode="nearest")


test_datagen = ImageDataGenerator(rescale=1 / 255.0)

With-Out Data Augmentation

In [13]:
batch_size = 32
train_batches = test_datagen.flow_from_directory(directory = train_dir,shuffle = True,
                                                   batch_size = batch_size)

val_batches = test_datagen.flow_from_directory(directory = valid_dir,shuffle = True,
                                                   batch_size = batch_size)

test_batches = test_datagen.flow_from_directory(directory= test_dir,shuffle = False,
                                                   batch_size = 1)

Found 17950 images belonging to 10 classes.
Found 2237 images belonging to 10 classes.
Found 2237 images belonging to 10 classes.


In [14]:
train_batches.class_indices

{'drinking': 0,
 'hair-and-makeup': 1,
 'operating_the_radio': 2,
 'reaching_behind': 3,
 'safe_driving': 4,
 'talking_on_the_phone-left': 5,
 'talking_on_the_phone-right': 6,
 'talking_to_passenger': 7,
 'texting-left': 8,
 'texting-right': 9}

In [15]:
len(train_batches.labels)

17950

In [16]:
cnn_model = models.Sequential()
cnn_model.add(layers.Conv2D(32,(3,3),activation = 'relu',name = 'Conv_input',input_shape = (256,256,3)))
cnn_model.add(layers.Conv2D(32,(3,3),activation = 'relu',name = 'Conv_2',padding = 'same'))
cnn_model.add(layers.Conv2D(32,(3,3),activation = 'relu',name = 'Conv_3',padding = 'same'))

cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.MaxPooling2D((2,2),name = 'max_1'))
cnn_model.add(layers.Conv2D(64,(3,3),activation = 'relu',name = 'Conv_4',padding='same'))
cnn_model.add(layers.Conv2D(64,(3,3),activation = 'relu',name = 'Conv_5',padding='same'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.MaxPooling2D((2,2),name = 'max_2'))

cnn_model.add(layers.Conv2D(128,(3,3),activation='relu'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.MaxPooling2D((2,2)))
cnn_model.add(layers.Conv2D(128,(3,3),activation='relu'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.Flatten())
cnn_model.add(layers.Dense(512,activation = 'relu',name = 'D1',))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.Dense(256,activation = 'relu',name = 'D2'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.Dense(256,activation = 'relu',name = 'D3'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.Dense(128,activation = 'relu' ,name ='D4'))
cnn_model.add(layers.BatchNormalization())

cnn_model.add(layers.Dense(10,activation = 'softmax',name = 'output'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
cnn_model.summary()

In [19]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1,patience=3)
history = cnn_model.fit(x = train_batches,
          steps_per_epoch=350,
          epochs=2,
          validation_data = val_batches,
          validation_steps= 50,
          callbacks=[es])

Epoch 1/2
350/350 ━━━━━━━━━━━━━━━━━━━━ 1295s 4s/step - accuracy: 0.2765 - loss: 2.0324 - val_accuracy: 0.2731 - val_loss: 2.0593
Epoch 2/2
211/350 ━━━━━━━━━━━━━━━━━━━━ 8:11 4s/step - accuracy: 0.4852 - loss: 1.4178

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


350/350 ━━━━━━━━━━━━━━━━━━━━ 772s 2s/step - accuracy: 0.5216 - loss: 1.3176 - val_accuracy: 0.5131 - val_loss: 1.3847


In [ ]:
test_loss, test_acc = cnn_model.evaluate(test_batches)
print(test_loss, test_acc)

In [ ]:
plot_model(cnn_model)

In [24]:
#Now let's apply our model onto Test data
test_datagen = ImageDataGenerator(rescale=1. / 255)
test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(256, 256),
        batch_size=batch_size,
        class_mode=None,
        shuffle=False)
#print(test_generator.filenames)
probabilities = cnn_model.predict(test_generator, 40//(batch_size))
#probabilities = model.predict_generator(test_generator, nb_test_samples//(batch_size-5))

Found 2237 images belonging to 10 classes.
70/70 ━━━━━━━━━━━━━━━━━━━━ 31s 437ms/step


In [ ]:
import pandas as pd
headers=['c0','c1','c2','c3','c4','c5','c6','c7','c8','c9']
df = pd.DataFrame(probabilities, columns=headers)
print(df)
#probabilities = (np.print(probabilities)).astype(int)


In [ ]:
mapper = []
i = 0
for file in test_generator.filenames:
    id = file.split("_")[1].split(".")[0]
    #id = int(file.split('_')[1].split('.')[0])
    print(id)
    mapper.append(id)
    i += 1

df['img'] = mapper

In [36]:
df = df.reindex(['img','c0','c1','c2','c3','c4','c5','c6','c7','c8','c9'],axis = 1)
#od = collections.OrderedDict(sorted(mapper.items()))
#tmp = pd.DataFrame({'id':list(mapper.keys()),'label':list(mapper.values())})
df.sort_values(by=['img'],inplace=True)
df['img'] = 'img_'+df['img'].astype(str)+'.jpg'
os.getcwd()
df.to_csv('submission_VijaNaar.csv', index=False)